In [ ]:
from ibl_schema.schema import *
import matplotlib.pyplot as plt
import torch

%load_ext autoreload
%autoreload 2

In [ ]:
X = np.load('Xtest.npy')
Y = np.load('Ytest.npy')

In [ ]:
# TODO: right now, everything gets binned and then each bin has a basis.
# need to provide the option to not bin and convolve on the continuous signal.
# the obvious case here is continuous position data
# can set binsize to None perhaps?

In [ ]:
from damn.basis_functions import *
bw = 5/1000
bf = raised_cosine_basis(10, .4, 0, bw, log_scale=False)
stimpost = raised_cosine_basis(10, 0, .4, bw, log_scale=False)
#bf2 = raised_cosine_basis(3, 0, 1, 1, log_scale=False)
#bf3 = raised_cosine_basis(3, 2, 2, 1, log_scale=False)
gauss = gaussian_smooth(1,1,.1)
gauss2 = gaussian_smooth(0,.5,bw)
position = raised_cosine_basis(5,3,3, .1)
#gauss = gaussian_smooth(.5, .5, 1)
identity = np.array([[1]])
#plt.plot(stimpost.T)
#plt.plot(gauss.T)
plt.plot(position.T)
plt.show()

# stack basis functions into a list
bfs = [stimpost, gauss, gauss]
bfs_no_time = [identity, gauss, gauss]
#bfs_no_time = [gauss2, gauss, gauss]
#bfs_no_time = [identity, position, position]

In [ ]:
# generate sample stimulus train, it should have an xy position and an intentity and a time
#stimulus_train = np.zeros((20, 3))
#stimulus_train[:, 0] = np.arange(20) # time
#stimulus_train[:, 1] = np.random.rand(20)*5 # x position
#stimulus_train[:, 2] = np.random.rand(20)*10 # y position

stimulus_train = np.load('demo_stim.npy')
stimulus_values = np.load('demo_stim_vals.npy')

# make the intensity randomly nonzero for 100 different timepoints
#stimulus_train[:, 3] =  (np.random.rand(1000) < 0.1).astype(float) # intensity
#stimulus_values = stimulus_values * 10 # intensity
# generate sample spike train, it should have a time and a neuron id
plt.imshow(stimulus_train[:100,1:], aspect='auto', interpolation='none')
print(stimulus_train.shape, stimulus_values.shape)

In [ ]:
# we now have (n_events, n_feature_dims) stimulus with stimulus_values
# we also know the binsize for each feature, so lets place these stimulus values into the appropriate bins
# build the tensor where ndims = n_feature_dims
feature_binsizes = [100/1000, 1, 1] # time bins of 5 ms, position bins of 1 pixel
# we need to know the number of bins for each feature dimension
# Define bin edges for each dimension
feature_val_ranges = [[0,20], 
                      [0,5],
                      [0,10]]

edges = []
for i in range(len(feature_binsizes)):
    start, stop = feature_val_ranges[i]
    binsize = feature_binsizes[i]
    
    nbins = int(np.floor((stop - start) / binsize))
    edges.append(np.linspace(start, start + nbins * binsize, nbins + 1))

# Compute N-D histogram with weights
feature_tensor, _ = np.histogramdd(
    stimulus_train,
    bins=edges,
    weights=stimulus_values
)

feature_tensor.shape

In [ ]:
stiminds = np.where(np.any(feature_tensor > 0, axis=(1,2)))[0]
xinds = np.where(np.any(feature_tensor > 0, axis=(0,2)))[0]
yinds = np.where(np.any(feature_tensor > 0, axis=(0,1)))[0]

In [ ]:
for i,(t,x,y) in enumerate(zip(stiminds, xinds, yinds)):
    print(stimulus_train[i])
    print(t)
    print(np.where(feature_tensor[t,:,:]))

    plt.imshow(feature_tensor[t,:,:], interpolation='none')
    plt.show()
    if i == 2:
        break

In [ ]:
import torch
import torch.nn.functional as F

import torch
import torch.nn.functional as F
import numpy as np

def _conv1d_along_axis(x, kernel, axis):
    """
    Convolve 1D kernels along a specific axis of an N-D tensor.

    x: np.ndarray, shape (D1, D2, ..., Dn)
    kernel: np.ndarray, shape (n_kernels, kernel_size)
    axis: int, axis along which to convolve

    Returns:
        out: np.ndarray, shape (..., n_kernels) where kernel dim is added at the end
    """
    x = torch.from_numpy(x).float()
    kernel = torch.from_numpy(kernel).float()  # (n_kernels, kernel_size)

    # 1️⃣ Move target axis to last dimension
    permute_order = [i for i in range(x.ndim) if i != axis] + [axis]
    x = x.permute(*permute_order)
    orig_shape = x.shape  # (..., L)
    batch = int(np.prod(orig_shape[:-1]))
    L = orig_shape[-1]

    # 2️⃣ Flatten all other dims into batch, add channel dim
    x = x.reshape(batch, 1, L)  # (batch, 1, L)

    # 3️⃣ Prepare kernel for conv1d
    n_kernels, K = kernel.shape
    kernel = torch.flip(kernel, dims=[1]) # need to flip for true convolution
    kernel = kernel.unsqueeze(1)  # (n_kernels, 1, K)
    #pad = K // 2  # SAME convolution

    # 4️⃣ Convolve
    out = F.conv1d(x, kernel, padding='same')  # (batch, n_kernels, L)
    # now shape: (batch, n_kernels, L)

    # 5️⃣ Move L to last dimension
    out = out.permute(0, 2, 1)  # (batch, L, n_kernels)

    # 6️⃣ Reshape back to original spatial dimensions + L + n_kernels
    out = out.reshape(*orig_shape[:-1], L, n_kernels)  # (..., L, n_kernels)

    # 7️⃣ Move the L dimension back to the original axis
    # Current shape: (D1, ..., Dn_except_axis, L, n_kernels)
    dims = list(range(out.ndim))
    dims.insert(axis, dims.pop(-2))  # move L into original axis
    out = out.permute(*dims)  # final shape: (..., n_kernels) at the end

    return out.numpy()



convd =  _conv1d_along_axis(feature_tensor, bf, axis=0)

# test case with no kernel
bf_test = np.array([[1]])
convd_test =  _conv1d_along_axis(feature_tensor, bf_test, axis=0)




In [ ]:
print(feature_tensor.shape,convd.shape)
t = 0
x = 0
y = 5

t = 20
x = 3
y = 5
plt.imshow(convd_test[:,x,y,:], aspect='auto')
plt.show()
plt.imshow(convd[:,x,y,:], aspect='auto')

In [ ]:
def convolve_nd(x, kernels):
    """
    Convolve a list of 1D kernels along each axis of an N-D tensor.

    x: np.ndarray, shape (D1, D2, ..., Dn)
    kernels: list of np.ndarray, each of shape (n_kernels_i, kernel_size_i)

    Returns:
        out: np.ndarray, shape (..., n_kernels_1, n_kernels_2, ..., n_kernels_n)
    """
    out = x
    for axis, kernel in enumerate(kernels):
        print(out.shape)
        out = _conv1d_along_axis(out, kernel, axis)
    print(out.shape)
    return out

#convd_all = convolve_nd(feature_tensor, bfs)
convd_all = convolve_nd(feature_tensor, bfs_no_time)
bfs_test = [bf_test, bf_test, bf_test]
convd_all_test = convolve_nd(feature_tensor, bfs_test)

In [ ]:
time_basis_ind = 0
time_offset = 0
plt.imshow(convd_all[t+time_offset,:,:,time_basis_ind,0,0], aspect='auto')
plt.show()
plt.imshow(feature_tensor[t,:,:], aspect='auto')
plt.show()
plt.imshow(convd_all_test[t,:,:,0,0,0], aspect='auto')
plt.show()

In [ ]:
#conv_flat = convd_all.reshape(convd_all.shape[0], -1)
feature_tensor.shape, convd_all.shape,
# now we need to reshape this convolved tensor into a 2D array where rows are timepoints and columns are features
n_timepoints = convd_all.shape[0]
n_features = np.prod(convd_all.shape[1:]) # all dimensions except time
conv_flat = convd_all.reshape(n_timepoints, n_features)
conv_flat.shape

In [ ]:
plt.imshow(conv_flat[:100,:], aspect='auto')